# Robustness Check: TWFE with Time-Varying AI Exposure

This notebook is a **robustness specification** for the primary TWFE model.

**Difference from the main model:** `ai_exposure` is measured contemporaneously (the occupation-level AI exposure in each observation year) rather than fixed to each individual's 2020 baseline. This allows the treatment intensity to update when an individual changes occupation.

**Why this is the robustness, not the main, specification:**
Approximately 37.5% of individuals changed occupation during 2020–2024. For job-changers, the time-varying `ai_exposure` conflates two effects: (1) the wage return to AI exposure, and (2) the wage gains from switching to a different — potentially higher-paying — occupation. Since workers may select into high-AI occupations precisely because those roles offer higher wages, the time-varying estimate is subject to endogenous occupational sorting. The baseline-fixed model removes this channel.

**Interpretation of the comparison:**
- If the two specifications produce similar estimates, the occupational-switching channel is negligible.
- If estimates diverge (as they do here), the difference quantifies how much of the raw TWFE estimate is attributable to selective occupational mobility rather than to AI exposure itself.

Model:
$$
\log(wage_{it}) = \sum_{t=2021}^{2024} \beta_t (AI_{it} \times \mathbf{1}\{year=t\}) + \gamma'X_{it} + \mu_i + \lambda_t + \varepsilon_{it}
$$

where $AI_{it}$ is the contemporaneous occupation-level AI exposure (time-varying), $\mu_i$ are individual fixed effects, and $\lambda_t$ are year fixed effects. Clustered standard errors at the individual level. Baseline year: 2020.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm

pd.set_option('display.float_format', '{:.4f}'.format)
plt.style.use('ggplot')

In [ ]:
project_root = Path.cwd()
if project_root.name == 'output':
    project_root = project_root.parent

data_path = project_root / 'data' / 'clean' / '08_wages_ai_analysis_panel.csv'
coef_out  = project_root / 'output' / 'panel_twfe_robustness_timevarying_coefficients.csv'
summary_out = project_root / 'output' / 'panel_twfe_robustness_timevarying_summary.txt'

raw = pd.read_csv(data_path)
print('Rows:', len(raw))
print('People:', raw['person_id'].nunique())
print('Years:', sorted(raw['year'].dropna().unique().tolist()))

In [ ]:
def within_transform(df: pd.DataFrame, cols, entity_col: str) -> pd.DataFrame:
    return df[cols] - df.groupby(entity_col)[cols].transform('mean')

fe = raw.copy()
numeric_cols = ['year', 'log_wage', 'ai_exposure', 'state', 'edu']
for c in numeric_cols:
    fe[c] = pd.to_numeric(fe[c], errors='coerce')

required = ['person_id', 'year', 'log_wage', 'ai_exposure', 'state', 'edu']
fe = fe.dropna(subset=required).copy()
fe = fe[fe['year'].between(2020, 2024)].copy()
fe = fe.groupby('person_id').filter(lambda g: g['year'].nunique() >= 2).copy()

# Time-varying ai_exposure: uses each observation year's occupation-level AI score.
# Individuals who change occupation will have different ai_exposure values across years.
# This is the robustness specification; see main model for the baseline-fixed version.

# Recode edu: HILDA edhigh1 is inverse (1=postgrad highest, 9=yr11 lowest).
# Map to ordinal 1-7 where higher = more educated. edu=10 (still at school) dropped.
edu_map = {9: 1, 8: 2, 5: 3, 4: 4, 3: 5, 2: 6, 1: 7}
fe['edu_ord'] = fe['edu'].map(edu_map)
fe = fe.dropna(subset=['edu_ord']).copy()

for yr in [2021, 2022, 2023, 2024]:
    fe[f'ai_x_yr{yr}'] = fe['ai_exposure'] * (fe['year'] == yr).astype(int)

# State: categorical dummies (8 Australian states/territories → 7 dummies)
state_dummies = pd.get_dummies(fe['state'].astype(int), prefix='st', drop_first=True).astype(float)
state_dummies.index = fe.index

ai_cols = ['ai_x_yr2021', 'ai_x_yr2022', 'ai_x_yr2023', 'ai_x_yr2024']
all_x = pd.concat([fe[ai_cols + ['edu_ord']], state_dummies], axis=1)

y_within = within_transform(fe, ['log_wage'], 'person_id')['log_wage']
x_within = all_x - all_x.groupby(fe['person_id']).transform('mean')

year_fe = pd.get_dummies(fe['year'].astype(int), prefix='yr', drop_first=True).astype(float)
year_fe_within = year_fe - year_fe.groupby(fe['person_id']).transform('mean')

X = pd.concat([x_within, year_fe_within], axis=1).astype(float)

model = sm.OLS(y_within, X).fit(cov_type='cluster', cov_kwds={'groups': fe['person_id']})

print('Observations:', len(fe))
print('Individuals:', fe['person_id'].nunique())
model.summary()

In [ ]:
coef_rows = ['ai_x_yr2021', 'ai_x_yr2022', 'ai_x_yr2023', 'ai_x_yr2024']
coef_table = pd.DataFrame({
    'coef':    model.params[coef_rows],
    'std_err': model.bse[coef_rows],
    'p_value': model.pvalues[coef_rows],
    'ci_low':  model.conf_int().loc[coef_rows, 0],
    'ci_high': model.conf_int().loc[coef_rows, 1],
}).round(6)

coef_table.to_csv(coef_out)

note = (
    "Robustness Check: TWFE with Time-Varying AI Exposure\n"
    "ai_exposure is measured contemporaneously (updates when individual changes occupation).\n"
    "Controls: edu_ord (ordinal 1-7), state dummies (categorical, 7 indicators)\n"
    "Entity FE via within-transform; year FE via dummies; clustered SE at individual level\n"
    "Baseline year: 2020\n"
)
with summary_out.open('w', encoding='utf-8') as f:
    f.write(note)
    f.write(f'Observations: {len(fe):,}\n')
    f.write(f'Individuals: {fe["person_id"].nunique():,}\n\n')
    f.write('Key AI exposure x year coefficients:\n')
    f.write(coef_table.to_string())
    f.write('\n\nFull model summary:\n')
    f.write(str(model.summary()))

coef_table

In [ ]:
plot_df = coef_table.reset_index().rename(columns={'index': 'term'})
plot_df['year'] = plot_df['term'].str.extract(r'(\d{4})')[0].astype(int)

fig, ax = plt.subplots(figsize=(7, 4))
ax.errorbar(
    plot_df['year'],
    plot_df['coef'],
    yerr=[plot_df['coef'] - plot_df['ci_low'], plot_df['ci_high'] - plot_df['coef']],
    fmt='o-',
    capsize=4
)
ax.axhline(0, color='black', linewidth=1, linestyle='--')
ax.set_title('Robustness: TWFE Coefficients (Time-Varying AI Exposure)')
ax.set_xlabel('Year')
ax.set_ylabel('Coefficient (with 95% CI)')
plt.tight_layout()
plt.show()

## Interpretation and Comparison with Main Model

The time-varying specification produces a positive and significant coefficient in 2021 (+0.083, p < 0.001) and statistically insignificant coefficients in 2022–2024. This contrasts sharply with the main (baseline-fixed) model, which yields negative and significant coefficients in all four years.

**Why the estimates diverge.** The difference is attributable to endogenous occupational switching. In the time-varying model, individuals who move into high-AI-exposure occupations between 2020 and 2021 contribute a rising `ai_exposure` value alongside a wage gain from job-changing. Because workers tend to switch jobs when they can improve their pay, this upward selection inflates the 2021 coefficient. The baseline-fixed model blocks this channel by holding each person's AI exposure at their 2020 occupation level, isolating the wage trajectory of 2020 incumbents.

**Economic significance of the gap.** The divergence between +0.083 (time-varying) and −0.090 (baseline-fixed) in 2021 implies that the apparent positive AI wage premium in the time-varying model is entirely — and more than entirely — explained by selective occupational mobility into high-AI jobs, not by AI exposure raising wages for workers who remain in their 2020 occupation.

**Recommended use.** Report the baseline-fixed model as the primary result. Present this robustness specification in the appendix alongside a discussion of the occupational-switching mechanism to explain the sign reversal.

## Summary and Conclusions

This robustness notebook estimates a TWFE event-study model where each individual's AI exposure is measured **contemporaneously** — the occupation-level AI score updates whenever the individual changes occupation. The key findings are summarised below.

---

**1. What the robustness model finds**

The time-varying specification yields a **positive and statistically significant** 2021 coefficient (+0.083, p < 0.001) and statistically insignificant coefficients in 2022–2024. On its face, this suggests that workers in high-AI-exposure occupations earned higher wages in 2021 relative to 2020, with the premium fading thereafter.

**2. Why the robustness model differs from the primary model**

The primary model (baseline-fixed AI exposure) produces **negative and significant** coefficients in all four years (2021–2024), the opposite sign. The divergence is attributable to **endogenous occupational switching**:

- Approximately 37.5% of individuals changed occupation during 2020–2024.
- In the time-varying model, job-changers who moved *into* high-AI occupations simultaneously receive a higher AI exposure score and a wage gain from job-changing itself. Because workers tend to switch jobs when doing so improves their pay, this selection inflates the 2021 coefficient upward.
- The baseline-fixed model blocks this channel by anchoring each person's AI exposure to their 2020 occupation regardless of subsequent job changes, isolating the wage trajectory of 2020 incumbents.

**3. What the gap tells us**

The difference between +0.083 (robustness) and −0.090 (primary) in 2021 implies that the *entire* apparent positive AI wage premium in the time-varying model — and then some — is attributable to selective occupational mobility into high-AI roles, not to AI exposure raising wages within occupations. Once endogenous switching is controlled for, high-AI-exposure workers in 2020 actually experienced *lower* wage growth relative to low-AI-exposure workers in every subsequent year.

**4. Role of this notebook in the overall analysis**

| Model | AI exposure | 2021 β | Interpretation |
|---|---|---|---|
| Primary TWFE | Baseline-fixed (2020) | −0.090*** | Causal estimate, net of switching endogeneity |
| This robustness | Time-varying (contemporaneous) | +0.083*** | Inflated by endogenous occupational sorting |

This robustness specification should be reported in the appendix. The primary result is the baseline-fixed TWFE model in `primary_analysis.ipynb`. The sign reversal between the two specifications is itself informative: it quantifies the magnitude of the occupational-switching channel and validates the identification strategy of the main model.